# Analyse de la souscription : qui répond favorablement à la campagne ?  

## Objectif  
Ce second notebook exploite la donnée nettoyée dans `01_exploration.ipynb`  
Il cherche à répondre à la problématique du projet :  
**quels profils de clients souscrivent le plus à un dépôt à terme, et comment cibler plus efficacement les prochains appels ?**  

Les indicateurs retenus ici alimenteront le dashboard Streamlit (`app.py`).  

## Point de repère  
Le taux de souscription global est d'environ **11,3 %**. Chaque segment analysé est comparé à cette référence :  
un taux nettement supérieur désigne un segment à privilégier, un taux inférieur un segment à moins solliciter.  

## Méthode  
- `y` vaut 0 ou 1 : la **moyenne** de `y` sur un groupe donne directement son taux de souscription.  
- Chaque taux est affiché avec le **volume** de clients du groupe, car un taux calculé sur un petit effectif n'est pas interprétable.  
- La variable `duration` (durée de l'appel) est volontairement **exclue de l'explication** : elle n'est connue qu'après l'appel et ne peut donc pas servir à cibler en amont.  

## Plan du notebook  
1. Chargement de la donnée nettoyée et indicateurs globaux  
2. Profil du client : métier, âge, niveau d'études  
3. Historique avec la banque : résultat de la campagne précédente (`poutcome`)  
4. Moment du contact : mois et jour de la semaine  
5. Contexte économique : Euribor et indicateurs macro  
6. Effort commercial : nombre d'appels par client (`campaign`)  
7. Synthèse : les 5 ou 6 indicateurs retenus pour le dashboard  

## 1. Chargement et indicateurs globaux

In [1]:
import pandas as pd

df = pd.read_csv("../data/processed/bank_marketing_clean.csv")

In [ ]:
df.shape # Affiche la forme (lignes, colonnes)

(41172, 21)

In [3]:
df.groupby("month")["y"].mean() # Regroupe les lignes par mois et calcule la moyenne de 'y' pour le groupe

month
apr    0.204943
aug    0.106073
dec    0.489011
jul    0.090389
jun    0.105115
mar    0.505495
may    0.064366
nov    0.101463
oct    0.439331
sep    0.449123
Name: y, dtype: float64

In [4]:
df.groupby("month")["y"].agg(["mean", "count"]) # Calcul le taux de souscription et la moyenne

,mean,count
month,,
apr,0.204943,2630
aug,0.106073,6175
dec,0.489011,182
jul,0.090389,7169
jun,0.105115,5318
mar,0.505495,546
may,0.064366,13765
nov,0.101463,4100
oct,0.439331,717


In [ ]:
# Tri par ordre alphabétique et conversion en pourcentage
mois = df.groupby("month")["y"].agg(taux="mean", appels="count")
mois["taux"] = (mois["taux"] * 100).round(1)
mois

,taux,appels
month,,
apr,20.5,2630
aug,10.6,6175
dec,48.9,182
jul,9.0,7169
jun,10.5,5318
mar,50.5,546
may,6.4,13765
nov,10.1,4100
oct,43.9,717


In [6]:
# Range les mois dans l'ordre
ordre = ["mar", "apr", "may", "jun", "jul", "aug", "sep", "oct", "nov", "dec"]
mois = mois.reindex(ordre)
mois

,taux,appels
month,,
mar,50.5,546
apr,20.5,2630
may,6.4,13765
jun,10.5,5318
jul,9.0,7169
aug,10.6,6175
sep,44.9,570
oct,43.9,717
nov,10.1,4100


Le jeu de données ne contient aucun appel en janvier ni en février : la campagne ne couvre que 10 mois sur 12. Les comparaisons entre mois portent donc uniquement sur ces 10 mois, et sont à nuancer car les volumes sont très inégaux.  

### Observations : taux de souscription par mois

- Mars (50,5 %), septembre, octobre et décembre affichent des taux très supérieurs à la moyenne (11,3 %), mais sur de petits volumes d'appels : ces chiffres sont à interpréter avec prudence.
- Mai, juin, juillet, août et novembre concentrent la grande majorité des appels, avec des taux de 6 à 11 %. Mai, le mois le plus appelé, a le taux le plus faible.
- Le volume d'appels est donc inversement lié au taux de réussite : on a appelé beaucoup quand ça convertissait peu.
- Limite : aucun appel en janvier et février, et les mois mélangent plusieurs années (2008-2010), donc contexte économique et saison sont confondus.

In [7]:
poutcome = df.groupby("poutcome")["y"].agg(taux="mean", appels="count")
poutcome["taux"] = (poutcome["taux"] * 100).round(1)
poutcome.sort_values("taux", ascending=False)

,taux,appels
poutcome,,
success,65.1,1373
failure,14.2,4252
nonexistent,8.8,35547


In [8]:
job = df.groupby("job")["y"].agg(taux="mean", appels="count")
job["taux"] = (job["taux"] * 100).round(1)
job.sort_values("taux", ascending=False)

,taux,appels
job,,
student,31.4,875
retired,25.3,1718
unemployed,14.2,1014
admin.,13.0,10418
unknown,11.2,330
management,11.2,2923
technician,10.8,6739
self-employed,10.5,1421
housemaid,10.0,1060


### Observations : historique (`poutcome`) et métier (`job`)

- **`poutcome`** : les clients ayant déjà souscrit lors d'une campagne précédente (`success`) reconvertissent à 65,1 %, soit près de 6 fois le taux global (11,3 %), sur un groupe de 1 373 clients. À l'inverse, 86 % des appels ont visé des clients jamais contactés, qui convertissent à 8,8 %.
- **`job`** : les étudiants (31,4 %) et les retraités (25,3 %) se démarquent nettement, avec des volumes suffisants pour être interprétés. Les `blue-collar` (6,9 %) et les `services` (8,1 %) sont sous la moyenne tout en représentant plus de 13 000 appels.
- Les `unknown` de `job` (11,2 %) sont au niveau de la moyenne : leur conservation comme modalité à part est cohérente.

In [ ]:
# Création des tranches d'âge
df["tranche_age"] = pd.cut(
    df["age"],
    bins=[0, 25, 35, 45, 55, 65, 100],
    labels=["<25", "25-34", "35-44", "45-54", "55-64", "65+"],
    right=False
)

In [ ]:
df["tranche_age"].isna().sum() # vérifie que tout les âges sont compris dans une borne

np.int64(0)

In [ ]:
# Regroupe par taux de souscription en % et nombre d'appels par tranche
age = df.groupby("tranche_age", observed=True)["y"].agg(taux="mean", appels="count")
age["taux"] = (age["taux"] * 100).round(1)
age

,taux,appels
tranche_age,,
<25,24.0,1067
25-34,12.2,13683
35-44,8.7,13494
45-54,8.6,8701
55-64,13.6,3565
65+,47.3,662


### Observations : tranche d'âge

- Le taux de souscription suit une **courbe en U** : 24,0 % chez les moins de 25 ans, 47,3 % chez les 65 ans et plus, contre 8,6 à 8,7 % pour les 35-54 ans (moyenne globale : 11,3 %).
- Ces résultats recoupent ceux du métier : étudiants (31,4 %) et retraités (25,3 %).
- Les 35-54 ans concentrent plus de 22 000 appels (plus de la moitié) pour un taux inférieur à la moyenne : l'effort commercial s'est porté sur les tranches les moins réceptives.
- Limite : ce sont des associations, pas des causes. Les 65+ ne représentent que 662 clients, leur taux est à interpréter avec un peu de prudence.

In [13]:
df["nb_appels"] = df["campaign"].clip(upper=6) # création de la colonne plafonnée

In [14]:
df["nb_appels"].value_counts().sort_index() # vérifie que la colonne est bien plafonnée à 6

nb_appels
1    17634
2    10567
3     5339
4     2649
5     1599
6     3384
Name: count, dtype: int64

In [ ]:

effort = df.groupby("nb_appels")["y"].agg(taux="mean", appels="count")
effort["taux"] = (effort["taux"] * 100).round(1)
effort